# Plant Disease Classification Using Deep Learning
**Apple leaf disease – multi-label classification with MobileNetV2 transfer learning**
Dataset: Plant Pathology 2021 – FGVC8 · Classes: healthy, scab, frog eye leaf spot, rust, powdery mildew, complex

### How to run in Google Colab
1. `Runtime → Change runtime type → T4 GPU`
2. Accept the competition rules once at https://www.kaggle.com/c/plant-pathology-2021-fgvc8/rules (needed to download the data).
3. Create a Kaggle API token (Kaggle → Settings → API → *Create New Token*). New tokens start with `KGAT_`; older accounts get a `kaggle.json` file instead.
4. `Runtime → Run all`. When asked, paste the token (or press Enter and upload `kaggle.json`).

No Kaggle account handy? Set `DEMO_MODE = True` in the first code cell to run the whole pipeline on tiny synthetic data (only to check that everything works – the scores are meaningless).

## 1. Imports and configuration

In [ ]:
import os, sys, glob, json, random, zipfile, shutil, subprocess, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report

import tensorflow as tf

warnings.filterwarnings("ignore")

# ------------------------- configuration -------------------------
DEMO_MODE   = False        # True = tiny synthetic dataset, just to smoke-test the pipeline

SEED        = 42
IMG_SIZE    = 224          # 224 x 224 inputs (as in the abstract)
BATCH_SIZE  = 64
EPOCHS      = 5            # 5 epochs (as in the abstract)
LR          = 1e-3
DROPOUT     = 0.3
VAL_FRACTION = 0.15        # 18,632 images -> 15,837 train / 2,795 validation
THRESHOLD   = 0.5          # decision threshold; top class is used if none exceeds it

# Optional improvement from "Future Scope" (the abstract's baseline keeps the backbone frozen)
FINE_TUNE   = False
FT_EPOCHS   = 5
FT_LR       = 1e-5
FT_LAYERS   = 30           # number of final backbone layers to unfreeze

CLASSES = ["healthy", "scab", "frog_eye_leaf_spot", "rust", "powdery_mildew", "complex"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

WORK_DIR = "/content" if os.path.isdir("/content") else os.path.abspath(".")
DATA_DIR = os.path.join(WORK_DIR, "demo_data" if DEMO_MODE else "plant-pathology-2021-fgvc8")
if DEMO_MODE:
    EPOCHS, FT_EPOCHS = 1, 1

random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "none found (training will be slow - switch Runtime to T4 GPU)")

## 2. Get the dataset

In [ ]:
def find_file(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    return hits[0] if hits else None


def make_demo_data(root, n_train=300, n_test=3):
    """Tiny synthetic stand-in for the real dataset (pipeline smoke test only)."""
    os.makedirs(os.path.join(root, "train_images"), exist_ok=True)
    os.makedirs(os.path.join(root, "test_images"), exist_ok=True)
    rng = np.random.RandomState(0)
    combos = ["healthy", "scab", "frog_eye_leaf_spot", "rust", "powdery_mildew", "complex",
              "scab frog_eye_leaf_spot", "rust complex"]
    def save_img(path, label):
        base = np.array([40 + 25 * (CLASS_TO_IDX[label.split()[0]]), 120, 60], dtype=np.float32)
        arr = np.clip(base + rng.randn(200, 300, 3) * 40, 0, 255).astype(np.uint8)
        Image.fromarray(arr).save(path, quality=90)
    rows = []
    for i in range(n_train):
        lab = combos[rng.randint(len(combos))]
        name = f"demo_{i:04d}.jpg"
        save_img(os.path.join(root, "train_images", name), lab)
        rows.append((name, lab))
    pd.DataFrame(rows, columns=["image", "labels"]).to_csv(os.path.join(root, "train.csv"), index=False)
    test_rows = []
    for i in range(n_test):
        name = f"demo_test_{i}.jpg"
        save_img(os.path.join(root, "test_images", name), "healthy")
        test_rows.append((name, "healthy"))
    pd.DataFrame(test_rows, columns=["image", "labels"]).to_csv(os.path.join(root, "sample_submission.csv"), index=False)


def setup_kaggle_credentials():
    """Accepts a new-style KGAT_ API token (pasted or from Colab Secrets) or a classic kaggle.json."""
    import getpass
    kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
    if os.environ.get("KAGGLE_API_TOKEN") or os.path.exists(kaggle_json) or \
            (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return
    try:                                                   # Colab Secrets (key icon in the left sidebar)
        from google.colab import userdata
        tok = userdata.get("KAGGLE_API_TOKEN")
        if tok:
            os.environ["KAGGLE_API_TOKEN"] = tok.strip()
            return
    except Exception:
        pass
    tok = getpass.getpass("Paste your Kaggle API token (starts with KGAT_), or press Enter to upload kaggle.json: ").strip()
    if tok:
        os.environ["KAGGLE_API_TOKEN"] = tok
        return
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("No Kaggle credentials found.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded.")
    os.makedirs(os.path.dirname(kaggle_json), exist_ok=True)
    with open(kaggle_json, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json, 0o600)


def download_competition_data(dest):
    os.makedirs(dest, exist_ok=True)
    setup_kaggle_credentials()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle>=1.8.0"], check=True)  # new tokens need a recent CLI
    print("Downloading Plant Pathology 2021 - FGVC8 (this is a large download, please wait) ...")
    res = subprocess.run(["kaggle", "competitions", "download", "-c", "plant-pathology-2021-fgvc8", "-p", dest],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            "Kaggle download failed:\n" + res.stdout + res.stderr +
            "\nMost common cause: you have not accepted the competition rules at "
            "https://www.kaggle.com/c/plant-pathology-2021-fgvc8/rules")
    for z in glob.glob(os.path.join(dest, "*.zip")):
        print("Extracting", os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        os.remove(z)


if DEMO_MODE:
    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        make_demo_data(DATA_DIR)
    print("DEMO_MODE: synthetic data created in", DATA_DIR)
else:
    if find_file(DATA_DIR, "train.csv") is None:
        download_competition_data(DATA_DIR)
    else:
        print("Dataset already present in", DATA_DIR)

TRAIN_CSV = find_file(DATA_DIR, "train.csv")
assert TRAIN_CSV is not None, "train.csv not found - dataset download/extraction failed."
ROOT = os.path.dirname(TRAIN_CSV)
TRAIN_DIR = os.path.join(ROOT, "train_images")
TEST_DIR = os.path.join(ROOT, "test_images")
assert os.path.isdir(TRAIN_DIR), f"{TRAIN_DIR} not found."
print("Data root:", ROOT)

## 3. Load labels (multi-hot encoding)

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print("Labelled images:", len(df))

Y = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
for i, s in enumerate(df["labels"].astype(str)):
    for tag in s.split():
        if tag not in CLASS_TO_IDX:
            raise ValueError(f"Unexpected label '{tag}' in row {i}")
        Y[i, CLASS_TO_IDX[tag]] = 1.0

counts = pd.Series(Y.sum(axis=0).astype(int), index=CLASSES)
print("\nImages per class (a leaf can count in several classes):")
print(counts.to_string())
print("\nLeaves with more than one label:", int((Y.sum(axis=1) > 1).sum()))

counts.plot(kind="bar", color="seagreen", figsize=(7, 3.5), title="Class distribution")
plt.ylabel("images"); plt.tight_layout(); plt.show()

## 4. Pre-process images once (resize to 224 × 224)
The original photos are very large, so decoding them every epoch would be slow. They are resized once and cached as a `uint8` array on disk.

In [ ]:
def load_resize(path):
    """Open a JPEG, shrink it quickly, convert to RGB and resize to IMG_SIZE x IMG_SIZE."""
    with Image.open(path) as im:
        im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))      # fast JPEG down-scaling
        im = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        return np.asarray(im, dtype=np.uint8)


def build_image_array(paths, cache_file, n_workers=4):
    if os.path.exists(cache_file):
        arr = np.load(cache_file, mmap_mode="r")
        if arr.shape == (len(paths), IMG_SIZE, IMG_SIZE, 3):
            print("Using cached array:", cache_file)
            return arr
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, e.g. {missing[0]}")
    out = np.lib.format.open_memmap(cache_file, mode="w+", dtype=np.uint8,
                                    shape=(len(paths), IMG_SIZE, IMG_SIZE, 3))
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        for i, img in enumerate(tqdm(ex.map(load_resize, paths), total=len(paths), desc="Resizing")):
            out[i] = img
    out.flush()
    del out
    return np.load(cache_file, mmap_mode="r")


train_paths = [os.path.join(TRAIN_DIR, n) for n in df["image"]]
CACHE_FILE = os.path.join(WORK_DIR, f"train_{IMG_SIZE}_{len(df)}{'_demo' if DEMO_MODE else ''}.npy")
X = build_image_array(train_paths, CACHE_FILE)
print("Image array:", X.shape, X.dtype)

## 5. Train / validation split and batch generators

In [ ]:
all_idx = np.arange(len(df))
strat = df["labels"].astype(str)
if strat.value_counts().min() < 2:      # stratification impossible for very rare combinations
    strat = None
train_idx, val_idx = train_test_split(all_idx, test_size=VAL_FRACTION, random_state=SEED, stratify=strat)
val_idx = np.sort(val_idx)
print(f"Train: {len(train_idx)}  |  Validation: {len(val_idx)}")


class ArraySeq(tf.keras.utils.Sequence):
    """Serves batches from the cached image array (works with Keras 2 and Keras 3)."""
    def __init__(self, X, Y, idx, batch_size, shuffle=False, **kwargs):
        super().__init__(**kwargs)
        self.X, self.Y = X, Y
        self.idx = np.array(idx)
        self.bs = batch_size
        self.shuffle = shuffle
        if shuffle:
            np.random.shuffle(self.idx)

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def __getitem__(self, i):
        b = np.sort(self.idx[i * self.bs:(i + 1) * self.bs])      # sorted -> fast memmap reads
        x = np.asarray(self.X[b], dtype=np.float32)
        if self.Y is None:
            return x
        return x, np.asarray(self.Y[b], dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)


train_seq = ArraySeq(X, Y, train_idx, BATCH_SIZE, shuffle=True)
val_seq   = ArraySeq(X, Y, val_idx,   BATCH_SIZE, shuffle=False)      # used during training
val_x_seq = ArraySeq(X, None, val_idx, BATCH_SIZE, shuffle=False)     # images only, for predict()
Y_val = Y[val_idx]

## 6. Model: MobileNetV2 → Global Average Pooling → Dropout → 6 sigmoid units
Augmentation (random flip, rotation, zoom) is part of the model and is only active during training.

In [ ]:
def build_model():
    L = tf.keras.layers
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="leaf_image")
    x = L.RandomFlip("horizontal_and_vertical")(inputs)
    x = L.RandomRotation(0.1)(x)
    x = L.RandomZoom(0.1)(x)
    x = L.Rescaling(1.0 / 127.5, offset=-1.0)(x)          # MobileNetV2 expects inputs in [-1, 1]
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                             include_top=False, weights="imagenet")
    base.trainable = False                                # frozen ImageNet backbone
    x = base(x, training=False)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dropout(DROPOUT)(x)
    outputs = L.Dense(NUM_CLASSES, activation="sigmoid", name="disease_probs")(x)
    return tf.keras.Model(inputs, outputs, name="apple_leaf_mobilenetv2"), base


def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.BinaryAccuracy(name="bin_acc"),
                 tf.keras.metrics.AUC(name="auc", multi_label=True)],
    )


model, base_model = build_model()
compile_model(model, LR)
model.summary()

## 7. Train

In [ ]:
history = model.fit(train_seq, validation_data=val_seq, epochs=EPOCHS, verbose=1)
hist = {k: list(v) for k, v in history.history.items()}

if FINE_TUNE:
    print("\nFine-tuning the last", FT_LAYERS, "backbone layers ...")
    base_model.trainable = True
    for layer in base_model.layers[:-FT_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:                      # keep BatchNorm statistics frozen
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, FT_LR)                          # re-compile after changing trainable flags
    ft = model.fit(train_seq, validation_data=val_seq, epochs=FT_EPOCHS, verbose=1)
    for k, v in ft.history.items():
        hist.setdefault(k, []).extend(v)

## 8. Evaluate on the validation set

In [ ]:
def decode_predictions(probs, threshold=THRESHOLD):
    """Binary labels: prob > threshold; if no class exceeds it, assign the most probable class."""
    pred = (probs > threshold).astype(int)
    none = pred.sum(axis=1) == 0
    pred[none, probs[none].argmax(axis=1)] = 1
    return pred


def evaluate_multilabel(y_true, probs, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    pred = decode_predictions(probs, threshold)
    try:
        macro_auc = roc_auc_score(y_true, probs, average="macro")
    except ValueError:                                   # a class missing from y_true (tiny demo data)
        macro_auc = float("nan")
    return {
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, pred, average="micro", zero_division=0),
        "macro_roc_auc": macro_auc,
        "exact_match_acc": float((pred == y_true).all(axis=1).mean()),
        "per_class_f1": dict(zip(CLASSES, f1_score(y_true, pred, average=None, zero_division=0))),
    }, pred


val_probs = model.predict(val_x_seq, verbose=1)
assert val_probs.shape == (len(val_idx), NUM_CLASSES), val_probs.shape
results, val_pred = evaluate_multilabel(Y_val, val_probs)

print("\n=========== Validation results ===========")
print(f"Macro F1          : {results['macro_f1']:.3f}   (abstract: 0.725)")
print(f"Micro F1          : {results['micro_f1']:.3f}   (abstract: 0.743)")
print(f"Macro ROC-AUC     : {results['macro_roc_auc']:.3f}   (abstract: 0.930)")
print(f"Exact-match acc.  : {results['exact_match_acc']:.3f}   (abstract: 0.690)")
print("\nPer-class F1:")
for c, v in results["per_class_f1"].items():
    print(f"  {c:<20s} {v:.2f}")
print("\n", classification_report(Y_val.astype(int), val_pred, target_names=CLASSES, zero_division=0))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].plot(hist["loss"], marker="o", label="train"); ax[0].plot(hist["val_loss"], marker="o", label="val")
ax[0].set_title("Binary cross-entropy loss"); ax[0].set_xlabel("epoch"); ax[0].legend()

ax[1].plot(hist["auc"], marker="o", label="train"); ax[1].plot(hist["val_auc"], marker="o", label="val")
ax[1].set_title("ROC-AUC (Keras, multi-label)"); ax[1].set_xlabel("epoch"); ax[1].legend()

names = list(results["per_class_f1"].keys()); vals = list(results["per_class_f1"].values())
bars = ax[2].bar(names, vals, color="seagreen")
ax[2].set_ylim(0, 1); ax[2].set_title("Per-class F1 (validation)")
ax[2].tick_params(axis="x", rotation=45)
for b, v in zip(bars, vals):
    ax[2].text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## 9. Predict on the test images and write `submission.csv`

In [ ]:
def predict_paths(paths, batch_size=32):
    imgs = np.stack([load_resize(p) for p in paths]).astype(np.float32)
    return model.predict(imgs, batch_size=batch_size, verbose=0)


def probs_to_label_string(probs_row, threshold=THRESHOLD):
    pred = decode_predictions(probs_row[None, :], threshold)[0]
    return " ".join(c for c, p in zip(CLASSES, pred) if p)


SAMPLE_SUB = find_file(DATA_DIR, "sample_submission.csv")
if SAMPLE_SUB is not None and os.path.isdir(TEST_DIR):
    sub = pd.read_csv(SAMPLE_SUB)
    test_paths = [os.path.join(TEST_DIR, n) for n in sub["image"]]
    test_probs = predict_paths(test_paths)
    sub["labels"] = [probs_to_label_string(p) for p in test_probs]
    SUB_PATH = os.path.join(WORK_DIR, "submission.csv")
    sub.to_csv(SUB_PATH, index=False)
    print("Saved", SUB_PATH)
    display(sub) if "display" in globals() else print(sub)
else:
    print("Test images / sample_submission.csv not found - skipping submission file.")

## 10. Try it on a single leaf

In [ ]:
def predict_one(path):
    p = predict_paths([path])[0]
    plt.figure(figsize=(8, 3.2))
    plt.subplot(1, 2, 1); plt.imshow(Image.open(path).convert("RGB")); plt.axis("off")
    plt.title("Predicted: " + probs_to_label_string(p))
    plt.subplot(1, 2, 2); plt.barh(CLASSES, p, color="seagreen"); plt.xlim(0, 1)
    plt.axvline(THRESHOLD, color="red", ls="--"); plt.title("Class probabilities")
    plt.tight_layout(); plt.show()


predict_one(train_paths[int(val_idx[0])])

## 11. Save the model and results

In [ ]:
MODEL_PATH = os.path.join(WORK_DIR, "plant_disease_mobilenetv2.keras")
model.save(MODEL_PATH)
with open(os.path.join(WORK_DIR, "validation_results.json"), "w") as f:
    json.dump({k: (v if not isinstance(v, np.floating) else float(v)) for k, v in results.items()},
              f, indent=2, default=float)
print("Saved model ->", MODEL_PATH)

try:                                         # offer downloads when running in Colab
    from google.colab import files
    files.download(MODEL_PATH)
    if "SUB_PATH" in globals():
        files.download(SUB_PATH)
except ImportError:
    pass

## 12. Live web app (upload a photo, get the diagnosis instantly)
Run this cell after training. It starts a small website with a public link (valid while this Colab session stays open). Open the link, drop in a leaf photo, and the diagnosis appears immediately.

In [ ]:
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr
def nice(name):
    return name.replace("_", " ")


def diagnose(image):
    """Called automatically whenever a photo is uploaded."""
    if image is None:
        return "Upload a leaf photo to see the diagnosis.", {}
    img = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    x = np.asarray(img, dtype=np.float32)[None]
    probs = np.asarray(model(x, training=False))[0]
    pred = probs > THRESHOLD
    if not pred.any():                                   # nothing above threshold -> most probable class
        pred[int(probs.argmax())] = True
    found = [nice(c) for c, p in zip(CLASSES, pred) if p]
    headline = "## Healthy leaf" if found == ["healthy"] else "## Detected: " + ", ".join(found)
    return headline, {nice(c): float(p) for c, p in zip(CLASSES, probs)}


demo = gr.Interface(
    fn=diagnose,
    inputs=gr.Image(type="pil", label="Upload an apple leaf photo"),
    outputs=[gr.Markdown(label="Diagnosis"), gr.Label(num_top_classes=NUM_CLASSES, label="Probability per class")],
    title="Apple Leaf Disease Detector",
    live=True,                                           # predict immediately after upload
)
demo.launch(share=True, debug=False)                     # prints a public link you can open on any device